In [ ]:
import urllib.parse
import json
import requests
from typing import Any, Optional
import uuid


In [ ]:
def invoke_endpoint(
    agent_arn: str,
    payload,
    session_id: str,
    bearer_token: Optional[str],
    region: str = "us-west-2",
    endpoint_name: str = "DEFAULT",
    stream: bool=False
) -> Any:
    """Invoke agent endpoint using HTTP request with bearer token."""
    escaped_arn = urllib.parse.quote(agent_arn, safe="")
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    }

    try:
        body = json.loads(payload) if isinstance(payload, str) else payload
    except json.JSONDecodeError:
        body = {"payload": payload}

    try:
        response = requests.post(
            url,
            params={"qualifier": endpoint_name},
            headers=headers,
            json=body,
            timeout=100,
            stream=stream,
        )
        if stream:
            last_data = False
            for line in response.iter_lines(chunk_size=1):
                if line:
                    line = line.decode("utf-8")
                    if line.startswith("data: "):
                        last_data = True
                        line = line[6:]
                        line = line.replace('"', "")
                        print(line)
                    elif line:
                        line = line.replace('"', "")
                        if last_data:
                            print("\n" + line)
                        last_data = False
        else:
            return response

    except requests.exceptions.RequestException as e:
        print("Failed to invoke agent endpoint: %s", str(e))
        raise

In [ ]:
from bedrock_agentcore.identity.auth import requires_access_token

agent_access_token = None

@requires_access_token(
    provider_name="agent-customer-support",
    scopes=[],  # Optional unless required
    auth_flow="M2M",
)
def _get_access_token_agent(access_token: str=""):
    global agent_access_token
    agent_access_token = access_token
    return access_token



In [ ]:

response = invoke_endpoint(
    agent_arn="arn:aws:bedrock-agentcore:us-west-2:407296935140:runtime/customernovpc-o8p5KsEHfM",
    payload={
        "prompt": "Hi"
    },
    session_id=str(uuid.uuid4()),
    bearer_token=_get_access_token_agent(),
)

In [ ]:
response.content

## Public VPC

In [ ]:
import boto3

In [ ]:
client_dp = boto3.client('bedrock-agentcore')
client_cp = boto3.client('bedrock-agentcore-control')

In [ ]:
response = client_cp.create_agent_runtime(
    agentRuntimeName='customersupportprivate',
    agentRuntimeArtifact={
        'containerConfiguration': {
            'containerUri': '407296935140.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-customernovpc:latest'
        }
    },
    roleArn='arn:aws:iam::407296935140:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-5ea7ad35e4',
    networkConfiguration={
        'networkMode': 'VPC',
        'networkModeConfig': {
            'subnets': ['subnet-058bd9653a44808d4'],
            'securityGroups': ['sg-0c955968f57a97fb3']
        }
    },
      protocolConfiguration={
        'serverProtocol': 'HTTP'
    },
    authorizerConfiguration={
        'customJWTAuthorizer': {
            'discoveryUrl': 'https://cognito-idp.us-west-2.amazonaws.com/us-west-2_2oTa6LAsj/.well-known/openid-configuration',
            'allowedClients': [
                '4qnrt1s3eoq8qv6v8r9afsqj4d',
            ]
        }
    },
)

In [ ]:
# import boto3
# agentcore_client = boto3.client(
#     'bedrock-agentcore',
#     region_name="us-west-2"
# )

# boto3_response = agentcore_client.invoke_agent_runtime(
#     agentRuntimeArn="arn:aws:bedrock-agentcore:us-west-2:407296935140:runtime/customersupportpublic-g9hq4aAV4S",
#     qualifier="DEFAULT",
#     payload=json.dumps({"prompt": "What is 2+2?"})
# )
# if "text/event-stream" in boto3_response.get("contentType", ""):
#     content = []
#     for line in boto3_response["response"].iter_lines(chunk_size=1):
#         if line:
#             line = line.decode("utf-8")
#             if line.startswith("data: "):
#                 line = line[6:]
#                 print(line)
#                 content.append(line)
#     display(Markdown("\n".join(content)))
# else:
#     try:
#         events = []
#         for event in boto3_response.get("response", []):
#             events.append(event)
#     except Exception as e:
#         events = [f"Error reading EventStream: {e}"]
#     display(Markdown(json.loads(events[0].decode("utf-8"))))

In [ ]:

response = invoke_endpoint(
    agent_arn="arn:aws:bedrock-agentcore:us-west-2:407296935140:runtime/customersupportprivate-IQM6oHCNlS",
    payload={
        "prompt": "Hi"
    },
    session_id=str(uuid.uuid4()),
    bearer_token=_get_access_token_agent(),
)

In [ ]:
response.content